# Chronos-2 energy fine-tune (GPU)

Generic fine-tune of `amazon/chronos-2` on the public UCI Appliances
Energy dataset via the `chronos` recipe in
powerdecksystems/powerdeck-models. Kaggle supplies the GPU
(~30 h/week free); the workflow in-repo covers CPU-feasible recipes.

Secrets needed in this kernel (Kaggle → Add-ons → Secrets):
`GITHUB_TOKEN` — classic PAT, `public_repo` + `workflow` on
powerdecksystems. `HF_TOKEN` optional.

In [ ]:
!pip install -q transformers onnx onnxruntime chronos-forecasting huggingface_hub pandas 'torchao>=0.16'
!git clone --depth 1 https://github.com/powerdecksystems/powerdeck-models /kaggle/working/pd-models
import os; os.chdir('/kaggle/working/pd-models')

In [ ]:
import json
# GITHUB_TOKEN arrives via the private 'powerdecksystems/secrets'
# dataset (attached in kernel-metadata) — Kaggle UI secrets are
# unavailable to API-pushed kernels.
with open('/kaggle/input/secrets/env.json') as f:
    os.environ['GITHUB_TOKEN'] = json.load(f)['GITHUB_TOKEN']
NAME = 'chronos-2-energy'
TAG  = 'models-finetune'
STEPS = '500'   # LoRA steps — bounded so a run stays well inside quota

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, 'tools/train.py',
    '--recipe', 'chronos', '--name', NAME, '--model', 'amazon/chronos-2',
    '--out', f'dist/{NAME}', '--epochs', '0', '--steps', STEPS,
    '--title', 'Chronos-2 (energy fine-tune)',
    '--description', 'Chronos-2 LoRA fine-tuned on UCI appliances energy data',
    '--use-case', 'Energy-domain forecasting',
    '--tier', 'ai', '--kind', 'forecast'])

In [ ]:
subprocess.check_call(['git','config','user.name','powerdeck-models-bot'])
subprocess.check_call(['git','config','user.email','bot@powerdecksystems.github'])
subprocess.check_call(['git','add','manifest.json'])
subprocess.check_call(['git','commit','-m',f'manifest: {NAME} @ {TAG} (kaggle)'])
# push via token remote — the clone is anonymous so a bare
# 'git push' has no credentials (v14 failure)
subprocess.check_call(['git','remote','set-url','origin',
                       f'https://x-access-token:{os.environ["GITHUB_TOKEN"]}'
                       f'@github.com/{repo}.git'])
subprocess.check_call(['git','push'])